# 3-Arms Technical Pipeline - OpenAI gpt-4o-mini

**Using OpenAI gpt-4o-mini for better rate limits and reliability**

## Configuration:
- Model: **gpt-4o-mini** (fast, cheap, reliable)
- API: OpenAI (https://api.openai.com/v1)
- Rate limit: 200 calls/min (much higher than Groq's 30 RPM)
- Concurrency: 8 (parallel processing)
- Cost: ~$0.80 for all 5,532 predictions

## Features:
- 3 arms: baseline, volume, full_technical
- Checkpoint saving every 50 results
- Resume capability
- Intelligent retry with exponential backoff

## Estimated Runtime:
- 5,532 total calls at 200 calls/min = **~28 minutes**

In [ ]:
import os
import re
import time
import asyncio
from datetime import datetime, timezone
import pandas as pd
from openai import AsyncOpenAI

# Config
DATA_PATH = "data/markets_microstructure_v2_v3_merged.csv"
OUT_DIR = "data/output"

# Output naming with timestamp
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_FILENAME = f"run_{RUN_TIMESTAMP}_3arms_openai_gpt4o_mini.csv"
OUT_PATH = os.path.join(OUT_DIR, OUT_FILENAME)
CHECKPOINT_PATH = OUT_PATH + ".checkpoint"

MODEL = "gpt-4o-mini"
TEMPERATURE = 0
CONCURRENCY = 8  # OpenAI can handle this
CHECKPOINT_INTERVAL = 50
MAX_RETRIES = 3

# Create output directory
os.makedirs(OUT_DIR, exist_ok=True)

# Get OpenAI API key (note: OPEN_AI_API_KEY with underscore)
api_key = os.environ.get("OPEN_AI_API_KEY")
if not api_key:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        api_key = os.environ.get("OPEN_AI_API_KEY")
    except:
        pass

if not api_key:
    api_key = input("Enter OPEN_AI_API_KEY: ").strip()

client = AsyncOpenAI(
    base_url="https://api.openai.com/v1",
    api_key=api_key,
)

print(f"{'='*70}")
print(f"3-ARMS TECHNICAL PIPELINE - OPENAI GPT-4O-MINI")
print(f"{'='*70}")
print(f"Input:       {DATA_PATH}")
print(f"Output:      {OUT_PATH}")
print(f"Checkpoint:  {CHECKPOINT_PATH}")
print(f"Model:       {MODEL}")
print(f"API:         OpenAI (https://api.openai.com/v1)")
print(f"Concurrency: {CONCURRENCY}")
print(f"Checkpoints: every {CHECKPOINT_INTERVAL} results")
print(f"{'='*70}")

In [ ]:
# Load data
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} markets")
print(f"Total API calls needed: {len(df) * 3} (markets × 3 arms)")
print(f"Estimated time at 200 calls/min: {(len(df) * 3) / 200:.1f} minutes")
print(f"Estimated cost: ~$0.80 (at $0.15 per 1M input tokens)")
df.head(2)

In [ ]:
# Prompt templates
SYSTEM_PROMPT = """You are forecasting the probability this market resolves YES.
Treat mid_yes as a prior probability, then update/reconsider it using your reasoning of the additional signals provided for this branch to bias your output.
Output only one decimal number between 0 and 1 (example: 0.023). No words, no JSON, no punctuation."""

def build_baseline(row):
    return f"""You are forecasting the probability this market resolves YES.

Question: {row['event_title']}
mid_yes (prior): {row['mid_yes']:.2f}

Available signals: None (baseline - only question and prior).
Use your reasoning about the question to update the prior.
Output only the final decimal probability."""

def build_volume(row):
    return f"""You are forecasting the probability this market resolves YES.

Question: {row['event_title']}
mid_yes (prior): {row['mid_yes']:.2f}

Available signals: volume, volume_24h.
- volume: ${row['volume']:,.0f}
- volume_24h: ${row['volume_24h']:,.0f}

Update mid_yes using these signals: higher activity suggests more informed prior (smaller change); lower activity allows larger reconsideration.
Output only the final decimal probability."""

def build_full_technical(row):
    spread_text = f"{row['spread_yes']:.2f}" if pd.notna(row['spread_yes']) else "N/A"
    tick_text = f"{row['tick_size']:.2f}" if pd.notna(row['tick_size']) else "N/A"
    
    prompt = f"""You are forecasting the probability this market resolves YES.

Question: {row['event_title']}
mid_yes (prior): {row['mid_yes']:.2f}

Available signals: volume, volume_24h, open_interest, liquidity, spread_yes, tick_size, time_to_close_hours.
- volume: ${row['volume']:,.0f}
- volume_24h: ${row['volume_24h']:,.0f}
- open_interest: ${row['open_interest']:,.0f}
- liquidity: {row['liquidity']:.2f}
- yes_bid: {row['yes_bid']:.2f}
- yes_ask: {row['yes_ask']:.2f}
- spread_yes: {spread_text}
- last_price: {row['last_price']:.2f}
- tick_size: {tick_text}
- time_to_close_hours: {row['time_to_close_hours']:.1f}"""
    
    if pd.notna(row.get('return_24h')):
        prompt += f"""

Additional technical signals: return_1h, return_6h, return_24h, trend_slope_24h, max_drawdown_24h, volatility_24h, vol_regime_shift, high_low_range_24h.
- return_1h: {row['return_1h']:.3f}
- return_6h: {row['return_6h']:.3f}
- return_24h: {row['return_24h']:.3f}
- trend_slope_24h: {row['trend_slope_24h']:.3f}
- max_drawdown_24h: {row['max_drawdown_24h']:.3f}
- volatility_24h: {row['volatility_24h']:.3f}
- vol_regime_shift: {row['vol_regime_shift']:.3f}
- high_low_range_24h: {row['high_low_range_24h']:.3f}"""
    
    prompt += """

Update mid_yes using these: higher activity/tighter spread => trust prior more (smaller change); lower activity/wider spread/large ticks => allow larger reconsideration.
Output only the final decimal probability."""
    return prompt

ARMS = {
    'baseline': build_baseline,
    'volume': build_volume,
    'full_technical': build_full_technical,
}

print("Prompt templates loaded")

In [ ]:
# API functions with retry
import random

def parse_probability(text):
    """Extract probability from response"""
    if not text or not text.strip():
        raise ValueError("Empty response")
    match = re.search(r'([01]?\.\d+|[01])', text.strip())
    if not match:
        raise ValueError(f"No probability in: '{text}'")
    p = float(match.group(1))
    if p > 1:
        p = p / 100
    if not (0 <= p <= 1):
        raise ValueError(f"Out of range: {p}")
    return p

async def query_market(row, arm_name, semaphore):
    """Query one market with retry logic"""
    async with semaphore:
        prompt = ARMS[arm_name](row)
        raw = None
        p_yes = None
        error = None
        
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                response = await client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=TEMPERATURE,
                )
                raw = response.choices[0].message.content.strip()
                p_yes = parse_probability(raw)
                error = None
                break
                
            except ValueError as e:
                # Parsing error - don't retry
                error = f"Parse: {str(e)}"
                break
                
            except Exception as e:
                error = f"API: {str(e)}"
                if attempt < MAX_RETRIES:
                    # Exponential backoff
                    wait = (2 ** (attempt - 1)) + random.uniform(0, 1)
                    await asyncio.sleep(wait)
        
        return {
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'model': MODEL,
            'arm': arm_name,
            'event_ticker': row['event_ticker'],
            'market_ticker': row['market_ticker'],
            'title': row['event_title'],
            'mid_yes': row['mid_yes'],
            'raw_response': raw,
            'p_yes': p_yes,
            'error': error,
            'has_technical': pd.notna(row.get('return_24h')),
            'attempts': attempt
        }

print(f"API functions loaded (max {MAX_RETRIES} retries)")

In [ ]:
# Checkpoint management
def load_checkpoint():
    """Load existing checkpoint if it exists"""
    if os.path.exists(CHECKPOINT_PATH):
        df_checkpoint = pd.read_csv(CHECKPOINT_PATH)
        print(f"Found checkpoint with {len(df_checkpoint)} results")
        return df_checkpoint
    return None

def save_checkpoint(results):
    """Save checkpoint"""
    df_results = pd.DataFrame(results)
    df_results.to_csv(CHECKPOINT_PATH, index=False)

def get_completed_calls(checkpoint_df):
    """Get set of (market_ticker, arm) tuples already completed"""
    if checkpoint_df is None:
        return set()
    return set(zip(checkpoint_df['market_ticker'], checkpoint_df['arm']))

print("Checkpoint functions loaded")

In [ ]:
# Main pipeline
async def run_pipeline(df_markets):
    # Load checkpoint if exists
    checkpoint_df = load_checkpoint()
    completed_calls = get_completed_calls(checkpoint_df)
    
    # Start with checkpoint results or empty list
    results = checkpoint_df.to_dict('records') if checkpoint_df is not None else []
    
    # Build list of all calls to make
    all_calls = []
    for _, row in df_markets.iterrows():
        for arm_name in ARMS.keys():
            call_id = (row['market_ticker'], arm_name)
            if call_id not in completed_calls:
                all_calls.append((row, arm_name))
    
    total_calls = len(all_calls)
    already_completed = len(completed_calls)
    
    print(f"\n{'='*70}")
    print(f"OPENAI GPT-4O-MINI PIPELINE")
    print(f"{'='*70}")
    print(f"Already completed: {already_completed}")
    print(f"Remaining calls: {total_calls}")
    print(f"Total calls: {already_completed + total_calls}")
    print(f"Estimated time: {total_calls / 200:.1f} minutes (at ~200 calls/min)")
    print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*70}\n")
    
    if total_calls == 0:
        print("All calls already completed!")
        return pd.DataFrame(results)
    
    # Create semaphore for concurrency control
    semaphore = asyncio.Semaphore(CONCURRENCY)
    
    # Create all tasks
    tasks = [query_market(row, arm_name, semaphore) for row, arm_name in all_calls]
    
    # Process with progress tracking
    start_time = time.time()
    successful = sum(1 for r in results if r.get('error') is None)
    
    for i, task in enumerate(asyncio.as_completed(tasks), 1):
        result = await task
        results.append(result)
        
        if result['error'] is None:
            successful += 1
        
        # Save checkpoint periodically
        if i % CHECKPOINT_INTERVAL == 0 or i == total_calls:
            save_checkpoint(results)
        
        # Progress update every 100 calls
        if i % 100 == 0 or i == total_calls:
            elapsed = time.time() - start_time
            rate = i / (elapsed / 60) if elapsed > 0 else 0
            remaining_time = (total_calls - i) / rate if rate > 0 else 0
            total_completed = already_completed + i
            success_rate = successful / total_completed * 100 if total_completed > 0 else 0
            
            print(f"[{total_completed:4d}/{already_completed + total_calls}] "
                  f"Success: {successful}/{total_completed} ({success_rate:.1f}%) | "
                  f"Rate: {rate:.0f}/min | ETA: {remaining_time:.0f}m")
    
    elapsed = time.time() - start_time
    print(f"\n{'='*70}")
    print(f"COMPLETED")
    print(f"{'='*70}")
    print(f"Total time: {elapsed/60:.1f} minutes")
    print(f"Total calls: {already_completed + total_calls}")
    print(f"Successful: {successful}/{already_completed + total_calls} ({successful/(already_completed + total_calls)*100:.1f}%)")
    print(f"Finished: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*70}\n")
    
    return pd.DataFrame(results)

print("Pipeline function loaded")
print("\nReady to run! Execute the next cell to start.")

In [ ]:
# Run the pipeline
df_results = await run_pipeline(df)

# Save final results
df_results.to_csv(OUT_PATH, index=False)
print(f"\nSaved final results to: {OUT_PATH}")

# Clean up checkpoint
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print(f"Removed checkpoint file")

In [ ]:
# Summary statistics
total = len(df_results)
success = df_results['error'].isna().sum()
errors = df_results['error'].notna().sum()

print(f"\n{'='*70}")
print(f"FINAL SUMMARY")
print(f"{'='*70}")
print(f"Total predictions: {total:,}")
print(f"Successful: {success:,} ({success/total*100:.1f}%)")
print(f"Errors: {errors:,} ({errors/total*100:.1f}%)")

print(f"\nBy arm:")
for arm in ['baseline', 'volume', 'full_technical']:
    df_arm = df_results[df_results['arm'] == arm]
    arm_success = df_arm['error'].isna().sum()
    print(f"  {arm:15s}: {arm_success:4,} / {len(df_arm):,} ({arm_success/len(df_arm)*100:.1f}%)")

df_full = df_results[df_results['arm'] == 'full_technical']
with_tech = df_full['has_technical'].sum()
print(f"\nMarkets with technical data: {with_tech:,} / {len(df_full):,} ({with_tech/len(df_full)*100:.1f}%)")

if errors > 0:
    print(f"\nTop error types:")
    error_counts = df_results[df_results['error'].notna()]['error'].value_counts().head(5)
    for error, count in error_counts.items():
        print(f"  {error[:60]:60s}: {count:,}")

print(f"\nSample successful predictions:")
display(df_results[df_results['error'].isna()].head(10)[['event_ticker', 'arm', 'p_yes', 'raw_response', 'has_technical']])